# LeWorldModel (LeWM) on Push-T — from-scratch replication

Runs on a **free Colab T4 GPU**. Trains LeWM from scratch on the Push-T
expert dataset (downloaded as **HDF5** from HuggingFace), validates the
success rate with CEM planning in the real Push-T env, then produces
decoded dream rollouts and a t-SNE of the latent space.

> Runtime → Change runtime type → **T4 GPU** before running.

Notes on the free-tier budget are in the final cell; defaults
(img=112, epochs=20, batch=96) are tuned to finish inside one
session. Scale them up toward the paper config on a bigger GPU.

## 0. Check the GPU (must be CUDA — the repo is CUDA-only)

In [ ]:
!nvidia-smi -L
import torch; assert torch.cuda.is_available(), 'Enable a GPU runtime!'
print('CUDA device:', torch.cuda.get_device_name(0))
# T4 is Turing -> supports fp16 but NOT bf16; we use 16-mixed below.

## 1. Install LeWM + stable-worldmodel and clone the paper repo

In [ ]:
%%bash
set -e
pip -q install 'stable-worldmodel[train,env]' zstandard imageio 2>&1 | tail -2
# Push-T is a pymunk/pygame env -> needs a headless framebuffer for eval;
# zstd decompresses the dataset archive.
apt-get -qq install -y xvfb zstd > /dev/null
cd /content && rm -rf le-wm && git clone -q https://github.com/lucas-maes/le-wm.git
echo 'installed.'

In [ ]:
import os, sys
os.environ['STABLEWM_HOME'] = '/content/.stable-wm'
os.makedirs(os.environ['STABLEWM_HOME'], exist_ok=True)
os.chdir('/content/le-wm'); sys.path.insert(0, '/content/le-wm')
print('STABLEWM_HOME =', os.environ['STABLEWM_HOME'])

## 2. Download the Push-T **HDF5** dataset from HuggingFace

`quentinll/lewm-pusht` ships `pusht_expert_train.h5.zst` (~13 GB). We
decompress it to `$STABLEWM_HOME` (used directly by evaluation), then
convert a copy to the compact Lance table (~0.8 GB) that training reads
for fast random access.

In [ ]:
# download the HDF5 archive (~13 GB) via the stable hub API
import os
from huggingface_hub import hf_hub_download
home = os.environ['STABLEWM_HOME']
p = hf_hub_download('quentinll/lewm-pusht', 'pusht_expert_train.h5.zst',
                    repo_type='dataset', local_dir=home)
print('downloaded ->', p)

In [ ]:
%%bash
set -e
cd $STABLEWM_HOME
echo 'decompressing (a few minutes)...'
zstd -d -f pusht_expert_train.h5.zst -o pusht_expert_train.h5
rm -f pusht_expert_train.h5.zst   # reclaim disk
ls -lh pusht_expert_train.h5

In [ ]:
# HDF5 -> Lance (fast training format). Keeps the .h5 for eval.
import os
from stable_worldmodel.data import convert
src = os.path.join(os.environ['STABLEWM_HOME'], 'pusht_expert_train.h5')
dst = os.path.join(os.environ['STABLEWM_HOME'], 'pusht_expert_train.lance')
convert(src, dst, dest_format='lance', mode='overwrite')
print('converted ->', dst)

In [ ]:
!swm datasets   # sanity-check: should list pusht_expert_train (Lance + HDF5)

## 3. Train LeWM **from scratch** on Push-T

Uses the paper repo's `train.py` (its real `jepa.py` model + SIGReg +
next-embedding loss). Overrides adapt the CUDA/bf16 defaults to a free T4
and a single-session budget. This is genuine from-scratch training —
`encoder.pretrained=false` in the model config.

In [ ]:
%%bash
cd /content/le-wm
export STABLEWM_HOME=/content/.stable-wm
python train.py \
  data=pusht \
  img_size=112 \
  trainer.max_epochs=20 \
  trainer.precision=16-mixed \
  trainer.accelerator=gpu trainer.devices=1 \
  loader.batch_size=96 \
  num_workers=2 \
  output_model_name=pusht/lewm

In [ ]:
!swm checkpoints pusht   # confirm weights_epoch_*.pt were written

## 4. Validate the success rate (CEM planning in the real Push-T env)

Replays expert start/goal pairs and plans with the trained world model.
`metrics` reports the **success rate** (fraction of episodes solved).
Run under `xvfb` because Push-T renders through pygame.

In [ ]:
%%bash
cd /content/le-wm
export STABLEWM_HOME=/content/.stable-wm
xvfb-run -a python eval.py --config-name=pusht.yaml \
  policy=pusht/lewm/weights_epoch_20.pt \
  eval.img_size=112 \
  eval.num_eval=20 \
  solver.device=cuda
# success rate is printed to stdout above as `metrics`, and appended to:
echo '--- results file ---'; tail -n 20 $STABLEWM_HOME/pusht/lewm/pusht_results.txt 2>/dev/null || true

## 5. Save the visualization scripts

The paper's JEPA has no decoder, so the decoded dream rollouts use a
small pixel decoder trained *after* training on the **frozen** latents
(a probe — it never touches the world-model weights).

In [ ]:
%%writefile /content/le-wm/decoder.py
"""A small pixel decoder for visualizing LeWM latents.

LeWM is a JEPA: it has NO decoder, because it never reconstructs pixels during
training. To *visualize* what a latent (or a dreamed/predicted latent) "looks
like", we train a lightweight decoder AFTER the fact, on top of the FROZEN
encoder+projector, to map a latent embedding back to an image. This is exactly
the probe-decoder trick the paper uses for its decoded dream rollouts: the
decoder is never used to train the world model, only to peek into its latents.

Input : latent embedding of shape (B, D)   (D = projector output dim, e.g. 192)
Output: image of shape (B, 3, H, W)
"""

import torch
import torch.nn as nn


class PixelDecoder(nn.Module):
    """DCGAN-style transposed-conv decoder: latent vector -> RGB image.

    `img_size` must be a multiple of 16 (4 upsampling stages of x2 from a 4x4
    base ... actually 5 stages: 4->8->16->32->64->... ). We compute the number
    of upsampling blocks from img_size so it works for 64/112/128/224 etc.
    """

    def __init__(self, latent_dim: int = 192, img_size: int = 112, base_ch: int = 256):
        super().__init__()
        self.img_size = img_size

        # Start from a 4x4 feature map and upsample x2 until we reach img_size.
        # For sizes that are not powers of two (e.g. 112, 224) we upsample to
        # the next power of two >= img_size, then resize down at the end.
        start = 4
        target = 1
        while target < img_size:
            target *= 2
        self.render_size = target                     # power-of-two >= img_size
        n_up = 0
        s = start
        while s < target:
            s *= 2
            n_up += 1

        self.fc = nn.Linear(latent_dim, base_ch * start * start)
        self.base_ch = base_ch
        self.start = start

        layers = []
        ch = base_ch
        for i in range(n_up):
            out_ch = max(base_ch // (2 ** (i + 1)), 32)
            layers += [
                nn.ConvTranspose2d(ch, out_ch, 4, stride=2, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.GELU(),
            ]
            ch = out_ch
        self.up = nn.Sequential(*layers)
        self.to_rgb = nn.Conv2d(ch, 3, 3, padding=1)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.fc(z).view(-1, self.base_ch, self.start, self.start)
        x = self.up(x)
        x = torch.sigmoid(self.to_rgb(x))             # (B, 3, render_size, render_size)
        if self.render_size != self.img_size:
            x = nn.functional.interpolate(
                x, size=(self.img_size, self.img_size),
                mode="bilinear", align_corners=False,
            )
        return x


if __name__ == "__main__":
    for size in (64, 112, 128, 224):
        dec = PixelDecoder(latent_dim=192, img_size=size)
        z = torch.randn(2, 192)
        out = dec(z)
        n = sum(p.numel() for p in dec.parameters())
        print(f"img_size={size:3d} -> out {tuple(out.shape)}  params {n/1e6:.2f}M")


In [ ]:
%%writefile /content/le-wm/lewm_common.py
"""Shared helpers for the LeWM visualization scripts (run on Colab / a CUDA box).

These depend on the `stable_worldmodel` / `stable_pretraining` stack and the
paper repo (`le-wm`), so they are meant to run in the same environment where
you trained the model. The model-side tensor logic here is validated offline in
`_local_test.py` against the real `jepa.py`.
"""

import os
import sys
from pathlib import Path

import numpy as np
import torch

# Make the paper repo importable (jepa.py, module.py, utils.py).
LEWM_REPO = os.environ.get("LEWM_REPO", str(Path(__file__).resolve().parents[1] / "le-wm"))
sys.path.insert(0, LEWM_REPO)

import stable_pretraining as spt
import stable_worldmodel as swm
from utils import get_img_preprocessor, get_column_normalizer  # from le-wm/utils.py

# ImageNet stats used by the training img preprocessor (to invert for display).
IMAGENET_MEAN = torch.tensor(spt.data.dataset_stats.ImageNet["mean"]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor(spt.data.dataset_stats.ImageNet["std"]).view(1, 3, 1, 1)


def get_device():
    return "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")


def load_model(run_name, device=None):
    """Load a trained LeWM checkpoint. `run_name` is a path under
    $STABLEWM_HOME/checkpoints, e.g. 'pusht/lewm/weights_epoch_20.pt'."""
    device = device or get_device()
    model = swm.wm.utils.load_pretrained(run_name)
    model = model.to(device).eval()
    model.requires_grad_(False)
    model.interpolate_pos_encoding = True
    return model


def build_transform(dataset, img_size, norm_cols=("action", "proprio", "state")):
    """Reproduce the exact training transform: ImageNet-normalize + resize the
    image, z-score every low-dim column (fit on the dataset)."""
    tfs = [get_img_preprocessor(source="pixels", target="pixels", img_size=img_size)]
    for col in norm_cols:
        if col in dataset.column_names:
            tfs.append(get_column_normalizer(dataset, col, col))
    return spt.data.transforms.Compose(*tfs)


def load_clip_dataset(name, clip_len, frameskip=5, img_size=112, cache_dir=None,
                      keys=("pixels", "action", "proprio", "state")):
    """Load the Push-T dataset as contiguous clips of length `clip_len`."""
    keys = [k for k in keys]
    ds = swm.data.load_dataset(
        name, transform=None, cache_dir=cache_dir,
        num_steps=clip_len, frameskip=frameskip,
        keys_to_load=keys,
        keys_to_cache=[k for k in keys if not k.startswith("pixels")],
    )
    ds.transform = build_transform(ds, img_size)
    return ds


def denorm_image(x):
    """Invert ImageNet normalization for display. x: (..., 3, H, W) -> [0,1] numpy HWC."""
    x = x.detach().cpu().float()
    shape = x.shape
    x = x.reshape(-1, *shape[-3:])
    x = x * IMAGENET_STD + IMAGENET_MEAN
    x = x.clamp(0, 1)
    x = x.permute(0, 2, 3, 1).numpy()                 # (N, H, W, 3)
    return x.reshape(*shape[:-3], *x.shape[1:])


@torch.no_grad()
def encode_frames(model, pixels, device=None):
    """Encode frames to LeWM latents.
    pixels: (N, 3, H, W) normalized -> returns emb (N, D)."""
    device = device or get_device()
    info = {"pixels": pixels.unsqueeze(1).to(device)}   # (N, T=1, C,H,W)
    emb = model.encode(info)["emb"]                      # (N, 1, D)
    return emb[:, 0]


@torch.no_grad()
def dream_latents(model, context_pixels, actions, history_size=3, device=None):
    """Faithful open-loop dream: encode `history_size` context frames, then
    autoregressively predict future latents conditioned on `actions`.

    context_pixels: (history_size, 3, H, W)  normalized
    actions:        (T, act_in_dim)          normalized, T >= history_size
    returns:        predicted_emb (T', D)     (context + dreamed future)
    """
    device = device or get_device()
    pixels = context_pixels.unsqueeze(0).unsqueeze(0).to(device)  # (B=1,S=1,H,C,H,W)
    act = actions.unsqueeze(0).unsqueeze(0).to(device)            # (B=1,S=1,T,A)
    info = model.rollout({"pixels": pixels}, act, history_size=history_size)
    return info["predicted_emb"][0, 0]                            # (T', D)


In [ ]:
%%writefile /content/le-wm/train_decoder.py
"""Train a pixel decoder from scratch on FROZEN LeWM latents.

The world model stays frozen; we only learn decoder: latent -> image. This is
the probe decoder used for the decoded dream rollouts. Nothing here touches the
world model's weights, so it does not affect the from-scratch WM training.

Example:
    python train_decoder.py --model pusht/lewm/weights_epoch_20.pt \
        --img-size 112 --epochs 8 --out decoder_pusht.pt
"""

import argparse

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from decoder import PixelDecoder
from lewm_common import (get_device, load_model, load_clip_dataset,
                         denorm_image, IMAGENET_MEAN, IMAGENET_STD)


def to01(x):
    """normalized (B,3,H,W) -> [0,1] target for reconstruction."""
    return (x.cpu() * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="checkpoint path under $STABLEWM_HOME/checkpoints")
    ap.add_argument("--dataset", default="pusht_expert_train.lance")
    ap.add_argument("--img-size", type=int, default=112)
    ap.add_argument("--clip-len", type=int, default=4)
    ap.add_argument("--frameskip", type=int, default=5)
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--batch-size", type=int, default=64)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--max-steps", type=int, default=2000, help="cap steps/epoch for speed")
    ap.add_argument("--latent-dim", type=int, default=192)
    ap.add_argument("--out", default="decoder_pusht.pt")
    args = ap.parse_args()

    device = get_device()
    print(f"device={device}")

    model = load_model(args.model, device)
    ds = load_clip_dataset(args.dataset, args.clip_len, args.frameskip, args.img_size)
    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=True,
                        num_workers=0, drop_last=True)

    decoder = PixelDecoder(latent_dim=args.latent_dim, img_size=args.img_size).to(device)
    opt = torch.optim.AdamW(decoder.parameters(), lr=args.lr)

    for epoch in range(1, args.epochs + 1):
        decoder.train()
        running = seen = 0
        for step, batch in enumerate(loader, 1):
            if step > args.max_steps:
                break
            pixels = batch["pixels"].to(device)                 # (B,T,3,H,W)
            B, T = pixels.shape[:2]
            flat = pixels.reshape(B * T, *pixels.shape[2:])
            with torch.no_grad():
                emb = model.encode({"pixels": flat.unsqueeze(1)})["emb"][:, 0]
            recon = decoder(emb)                                # (B*T,3,H,W) in [0,1]
            target = to01(flat).to(device)
            loss = nn.functional.mse_loss(recon, target)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * flat.size(0); seen += flat.size(0)
        print(f"epoch {epoch}/{args.epochs}  recon_mse {running/seen:.5f}")
        torch.save({"state_dict": decoder.state_dict(),
                    "img_size": args.img_size, "latent_dim": args.latent_dim}, args.out)

    print(f"saved decoder -> {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/le-wm/dream_rollout.py
"""Produce decoded dream rollouts for a trained LeWM.

Given a real Push-T episode clip, we feed the first `history_size` frames as
context and let the world model imagine ("dream") future latents open-loop,
conditioned on the episode's real action sequence. We decode both the dreamed
latents and the ground-truth frames and lay them side by side.

Outputs a PNG grid (top row = ground truth, bottom row = decoded dream) and an
animated GIF.

Example:
    python dream_rollout.py --model pusht/lewm/weights_epoch_20.pt \
        --decoder decoder_pusht.pt --horizon 8 --episodes 3 --out dream.png
"""

import argparse

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch

from decoder import PixelDecoder
from lewm_common import (get_device, load_model, load_clip_dataset,
                         denorm_image, dream_latents)


def load_decoder(path, device):
    ckpt = torch.load(path, map_location="cpu")
    dec = PixelDecoder(latent_dim=ckpt["latent_dim"], img_size=ckpt["img_size"])
    dec.load_state_dict(ckpt["state_dict"])
    return dec.to(device).eval()


@torch.no_grad()
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--decoder", required=True)
    ap.add_argument("--dataset", default="pusht_expert_train.lance")
    ap.add_argument("--img-size", type=int, default=112)
    ap.add_argument("--frameskip", type=int, default=5)
    ap.add_argument("--history-size", type=int, default=3)
    ap.add_argument("--horizon", type=int, default=8, help="future steps to dream")
    ap.add_argument("--episodes", type=int, default=3, help="how many clips to render")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="dream.png")
    args = ap.parse_args()

    device = get_device()
    model = load_model(args.model, device)
    decoder = load_decoder(args.decoder, device)

    HS = args.history_size
    clip_len = HS + args.horizon
    ds = load_clip_dataset(args.dataset, clip_len, args.frameskip, args.img_size)

    rng = np.random.default_rng(args.seed)
    idxs = rng.choice(len(ds), size=args.episodes, replace=False)

    fig, axes = plt.subplots(2 * args.episodes, clip_len,
                             figsize=(1.4 * clip_len, 2.8 * args.episodes))
    if args.episodes == 1:
        axes = axes.reshape(2, clip_len)

    gif_frames = []
    for row, idx in enumerate(idxs):
        item = ds[int(idx)]
        pixels = item["pixels"]                       # (clip_len, 3, H, W) normalized
        actions = item["action"]                      # (clip_len, act_in)
        actions = torch.nan_to_num(actions, 0.0)

        gt = denorm_image(pixels)                     # (clip_len, H, W, 3)
        dreamed = dream_latents(model, pixels[:HS], actions,
                                history_size=HS, device=device)   # (T', D)
        dec_imgs = denorm_image(decoder(dreamed.to(device)))       # (T', H, W, 3)

        # align dreamed timeline to the clip (context frames shown as-is)
        for t in range(clip_len):
            ax_gt = axes[2 * row, t]; ax_dr = axes[2 * row + 1, t]
            ax_gt.imshow(gt[t]); ax_gt.axis("off")
            di = min(t, dec_imgs.shape[0] - 1)
            ax_dr.imshow(dec_imgs[di]); ax_dr.axis("off")
            if t < HS:
                ax_dr.set_title("ctx", fontsize=7)
        axes[2 * row, 0].set_ylabel("GT", fontsize=9)
        axes[2 * row + 1, 0].set_ylabel("dream", fontsize=9)

        # build a gif comparing gt vs dream over time for the first episode
        if row == 0:
            for t in range(clip_len):
                di = min(t, dec_imgs.shape[0] - 1)
                pair = np.concatenate([gt[t], dec_imgs[di]], axis=1)
                gif_frames.append((pair * 255).astype(np.uint8))

    fig.suptitle("LeWM decoded dream rollout — top: ground truth, bottom: imagined", fontsize=11)
    fig.tight_layout()
    fig.savefig(args.out, dpi=130, bbox_inches="tight")
    print(f"saved grid -> {args.out}")

    if gif_frames:
        gif_path = args.out.rsplit(".", 1)[0] + ".gif"
        imageio.mimsave(gif_path, gif_frames, duration=0.4)
        print(f"saved gif  -> {gif_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/le-wm/tsne_latents.py
"""t-SNE of the LeWM latent space on Push-T.

Encodes many frames sampled across episodes into LeWM latents, projects them to
2D with t-SNE, and colors points by physical quantities (block angle, agent
position, and episode progress). Clustering/structure by these quantities is
evidence the latent space encodes meaningful physical structure — the probing
result highlighted in the paper.

Example:
    python tsne_latents.py --model pusht/lewm/weights_epoch_20.pt \
        --num 3000 --img-size 112 --out tsne.png
"""

import argparse

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from lewm_common import get_device, load_model, load_clip_dataset, encode_frames


@torch.no_grad()
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--dataset", default="pusht_expert_train.lance")
    ap.add_argument("--img-size", type=int, default=112)
    ap.add_argument("--frameskip", type=int, default=5)
    ap.add_argument("--num", type=int, default=3000, help="frames to embed")
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--perplexity", type=float, default=30.0)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="tsne.png")
    args = ap.parse_args()

    device = get_device()
    model = load_model(args.model, device)

    # clip_len=1 gives single frames; keep state for coloring.
    ds = load_clip_dataset(args.dataset, clip_len=1, frameskip=args.frameskip,
                           img_size=args.img_size)
    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=True, num_workers=0)

    embs, states = [], []
    n = 0
    for batch in loader:
        pixels = batch["pixels"][:, 0]                    # (B,3,H,W)
        emb = encode_frames(model, pixels, device).cpu()
        embs.append(emb)
        if "state" in batch:
            states.append(batch["state"][:, 0].cpu())
        n += pixels.size(0)
        if n >= args.num:
            break

    Z = torch.cat(embs)[: args.num].numpy()
    state = torch.cat(states)[: args.num].numpy() if states else None
    print(f"embedded {Z.shape[0]} frames, dim {Z.shape[1]}")

    from sklearn.manifold import TSNE
    Y = TSNE(n_components=2, perplexity=args.perplexity, init="pca",
             random_state=args.seed).fit_transform(Z)

    # PushT state = [agent_x, agent_y, block_x, block_y, block_angle, ...]
    color_specs = [("progress (sample order)", np.arange(len(Y)))]
    if state is not None and state.shape[1] >= 5:
        color_specs = [
            ("block angle", state[:, 4]),
            ("block x", state[:, 2]),
            ("agent x", state[:, 0]),
        ]

    fig, axes = plt.subplots(1, len(color_specs), figsize=(5.2 * len(color_specs), 4.6))
    if len(color_specs) == 1:
        axes = [axes]
    for ax, (name, c) in zip(axes, color_specs):
        sc = ax.scatter(Y[:, 0], Y[:, 1], c=c, cmap="viridis", s=6, alpha=0.8)
        ax.set_title(f"LeWM latent t-SNE — colored by {name}", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(args.out, dpi=130, bbox_inches="tight")
    print(f"saved -> {args.out}")


if __name__ == "__main__":
    main()


## 6. Train the probe decoder (frozen world model)

In [ ]:
%%bash
cd /content/le-wm; export STABLEWM_HOME=/content/.stable-wm
python train_decoder.py --model pusht/lewm/weights_epoch_20.pt \
  --img-size 112 --epochs 8 --out /content/decoder_pusht.pt

## 7. Decoded dream rollouts

Feed 3 context frames, dream future latents open-loop under the real
action sequence, and decode. Top row = ground truth, bottom = imagined.

In [ ]:
%%bash
cd /content/le-wm; export STABLEWM_HOME=/content/.stable-wm
python dream_rollout.py --model pusht/lewm/weights_epoch_20.pt \
  --decoder /content/decoder_pusht.pt --img-size 112 \
  --horizon 8 --episodes 3 --out /content/dream.png

In [ ]:
from IPython.display import Image, display
display(Image('/content/dream.png'))
display(Image('/content/dream.gif'))

## 8. t-SNE of the latent space

Structure by block angle / positions is evidence the latents encode
physical state (the probing result from the paper).

In [ ]:
%%bash
cd /content/le-wm; export STABLEWM_HOME=/content/.stable-wm
python tsne_latents.py --model pusht/lewm/weights_epoch_20.pt \
  --img-size 112 --num 3000 --out /content/tsne.png

In [ ]:
from IPython.display import Image, display
display(Image('/content/tsne.png'))

## 9. Bundle the deliverables

In [ ]:
%%bash
cd /content
mkdir -p lewm_deliverables
cp -f dream.png dream.gif tsne.png lewm_deliverables/ 2>/dev/null || true
cp -f .stable-wm/pusht/lewm/pusht_results.txt lewm_deliverables/ 2>/dev/null || true
cp -rf .stable-wm/pusht/lewm/*.mp4 lewm_deliverables/ 2>/dev/null || true
cp -f .stable-wm/checkpoints/pusht/lewm/config.json lewm_deliverables/ 2>/dev/null || true
zip -qr lewm_deliverables.zip lewm_deliverables; ls -lh lewm_deliverables.zip

In [ ]:
from google.colab import files; files.download('/content/lewm_deliverables.zip')

## Free-tier budget notes

- Defaults: img=112px, 20 epochs, batch 96, fp16. Reaches a
  useful (not paper-maximal) success rate inside one free session.
- **Disk**: the decompressed `.h5` is large (tens of GB). Standard Colab
  disk (~100 GB) is enough; if you hit a limit, delete the `.h5` after
  the Lance convert and skip step 4 (eval needs the `.h5`).
- To approach the paper: `img_size=224`, `trainer.max_epochs=100`,
  `loader.batch_size=128`, on an A100/H200 (bf16). Everything else is the
  same command.